# Data Description
- Author: RUEI-HSIANG, CHANG（AI協助：Codex）
- Date first created: 2026-11-14
- Background: 補入缺漏社群帳號的候選人全範圍整合表。
- Unit of observation: 候選人280、候選人×平台840、來源觀察61、群別×平台12。
- Sample period: 登記候選人2026；舊來源2026-10–11，新來源2026-11-14；部分帳號可能歷史或失效。
- Known issues: 缺漏不等於無帳號；新目錄／百科外連未覆核，官方嵌入連結未測試帳號可用性；名稱歧義與複數URL需覆核。未宣稱全文人工覆核或活動／貼文完整性。
- Definition for each variable: candidate_key為原登記鍵；platform三平台；account_urls為JSON陣列；verified_account_urls僅保留舊輪已核對；institutional_source_account_urls是本輪官方來源擷取待目標核對；missing_before/after及newly_filled為0/1；source_urls與source_file/source_sha256追溯來源。wide保留每平台URL和狀態；summary記錄候選人總數、前後缺漏及新增。
- Data path / SHA-256 / Downstream analysis input: {"candidate_social_links_scope_20261114.csv": "ac9dab6e14504395106de57eb9d2861dc029e400a743448ec42e756098d68e89", "social_link_worklist_scope_20261114.csv": "0e70683029b03213783fecda1c8f77baa64887a758797711561abf2603f83b7f", "social_link_evidence_added_20261114.csv": "79e97555c46b0ee33f5670119d645ecdb36656ae690ad26b3970d05fcfa9b4ac", "social_link_missing_summary_20261114.csv": "d143ff8f961353a2353e28a5ceb1e99bd409dd70ce05ef35b2ae1831f9bc05b6"}
- Source data: 處理notebook PINS及 `data/raw/social_links_20261114/file_manifest.json`。
- Processing code: `data/code/prepare_social_links_scope_20261114.ipynb`
- Role: collection worklist / evidence / summary
- Verification status: clean kernel、保存檔hash與列數／鍵檢查。

## Change for Your Own Data
固定路徑與hash，不自動挑選最新檔。
## Summary
### Sample
### Summary Stats
### Assertions


第二輪更新：累計61筆來源觀察、填入55個候選人平台缺漏。本輪增加12個Facebook候選人連結，歷史目錄的連結狀態保留historical_unverified_lead；仍待確認帳號身分及可用性。新來源見 `data/raw/social_links_20261114_round2/file_manifest.json`。


In [1]:
from pathlib import Path
import csv,hashlib,json
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists())
FILES={'candidate_social_links_scope_20261114.csv': 'ac9dab6e14504395106de57eb9d2861dc029e400a743448ec42e756098d68e89', 'social_link_worklist_scope_20261114.csv': '0e70683029b03213783fecda1c8f77baa64887a758797711561abf2603f83b7f', 'social_link_evidence_added_20261114.csv': '79e97555c46b0ee33f5670119d645ecdb36656ae690ad26b3970d05fcfa9b4ac', 'social_link_missing_summary_20261114.csv': 'd143ff8f961353a2353e28a5ceb1e99bd409dd70ce05ef35b2ae1831f9bc05b6'}
for name,h in FILES.items():
 p=ROOT/'data/processed'/name
 assert hashlib.sha256(p.read_bytes()).hexdigest()==h
 with p.open(encoding='utf-8-sig',newline='') as f: rows=list(csv.DictReader(f))
 print(name,len(rows),'columns',list(rows[0]));print('First 10:',rows[:10])
 if 'worklist' in name:
  assert len(rows)==840 and len({(r['candidate_key'],r['platform']) for r in rows})==840
  assert sum(int(r['newly_filled']) for r in rows)==55
 if 'candidate_social_links' in name:assert len(rows)==280 and len({r['candidate_key'] for r in rows})==280
print('Assertions passed')


candidate_social_links_scope_20261114.csv 280 columns ['candidate_key', 'candidate_name', 'county_city', 'office_type', 'Facebook_urls', 'Facebook_status', 'Instagram_urls', 'Instagram_status', 'Threads_urls', 'Threads_status']
First 10: [{'candidate_key': '2026-C1-001', 'candidate_name': '郭璽', 'county_city': '臺北市', 'office_type': '直轄市長', 'Facebook_urls': '[]', 'Facebook_status': 'not_located', 'Instagram_urls': '[]', 'Instagram_status': 'not_located', 'Threads_urls': '[]', 'Threads_status': 'not_located'}, {'candidate_key': '2026-C1-002', 'candidate_name': '沈伯洋', 'county_city': '臺北市', 'office_type': '直轄市長', 'Facebook_urls': '["https://www.facebook.com/1087764201093899", "https://www.facebook.com/puma.taipei", "https://www.facebook.com/pumashen"]', 'Facebook_status': 'source_attribution_verified_previous_round', 'Instagram_urls': '["https://www.instagram.com/pumashen"]', 'Instagram_status': 'source_attribution_verified_previous_round', 'Threads_urls': '["https://www.threads.com/@pumash